# Scikit-Learn Mastery Notebook

Complete guide covering custom transformers, pipelines, algorithm comparison, tuning, and deployment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')
print('Setup complete')

## 1. Building Custom Transformers

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted, check_array

class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log(1+x) to skewed features."""
    def __init__(self, threshold=2.0):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        X = check_array(X)
        from scipy.stats import skew
        self.skewness_ = np.array([skew(X[:, i]) for i in range(X.shape[1])])
        self.skewed_cols_ = np.where(np.abs(self.skewness_) > self.threshold)[0]
        return self
    
    def transform(self, X):
        check_is_fitted(self)
        X = check_array(X, copy=True)
        X[:, self.skewed_cols_] = np.log1p(np.abs(X[:, self.skewed_cols_]))
        return X

# Test it
data = load_breast_cancer()
X, y = data.data, data.target
lt = LogTransformer(threshold=1.5)
X_transformed = lt.fit_transform(X)
print(f'Skewed columns: {lt.skewed_cols_}')
print(f'Original shape: {X.shape}, Transformed: {X_transformed.shape}')

## 2. Pipeline Composition

```
Raw Data → Impute → Scale → Feature Eng → Select → Model → Predictions
         ↓                                                              
    ColumnTransformer splits numeric/categorical                        
         ├── Numeric:  Impute(median) → Scale → LogTransform            
         └── Categorical: Impute(mode) → OneHotEncode                   
```

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Simple pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('log', LogTransformer(threshold=1.5)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print(f'Pipeline CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})')

## 3. ColumnTransformer for Real-World Data

In [ ]:
# Create mixed-type DataFrame for demo
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age': np.random.normal(40, 15, n),
    'income': np.random.exponential(50000, n),
    'gender': np.random.choice(['M', 'F'], n),
    'city': np.random.choice(['NYC', 'LA', 'CHI', 'HOU'], n),
    'target': np.random.binomial(1, 0.3, n)
})
# Add missing values
df.loc[np.random.choice(n, 30), 'age'] = np.nan
df.loc[np.random.choice(n, 20), 'income'] = np.nan

num_cols = ['age', 'income']
cat_cols = ['gender', 'city']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ]), cat_cols)
])

full_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

X_df = df.drop('target', axis=1)
y_df = df['target']
scores = cross_val_score(full_pipe, X_df, y_df, cv=5, scoring='roc_auc')
print(f'Mixed-type pipeline ROC-AUC: {scores.mean():.4f}')

## 4. Comparing 10 Algorithms on Same Dataset

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
import time

X_scaled = StandardScaler().fit_transform(X)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVM (RBF)': SVC(probability=True),
    'KNN (k=5)': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42, algorithm='SAMME'),
    'Extra Trees': ExtraTreesClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42),
}

results = []
for name, model in models.items():
    t0 = time.time()
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
    elapsed = time.time() - t0
    results.append({'Model': name, 'Accuracy': scores.mean(), 'Std': scores.std(), 'Time(s)': elapsed})

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(results_df['Model'], results_df['Accuracy'], xerr=results_df['Std'], color='steelblue')
ax.set_xlim(0.85, 1.0)
ax.set_xlabel('Accuracy')
ax.set_title('10 Algorithms Compared on Breast Cancer Dataset')
ax.axvline(x=results_df['Accuracy'].max(), color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning Visualization

In [ ]:
from sklearn.model_selection import validation_curve

param_range = np.arange(1, 50, 2)
train_scores, val_scores = validation_curve(
    RandomForestClassifier(random_state=42), X_scaled, y,
    param_name='max_depth', param_range=param_range,
    cv=5, scoring='accuracy', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(param_range, train_scores.mean(axis=1), 'o-', label='Training')
ax.plot(param_range, val_scores.mean(axis=1), 'o-', label='Validation')
ax.fill_between(param_range, val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.2)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Curve: Random Forest max_depth')
ax.legend()
ax.axvline(param_range[val_scores.mean(axis=1).argmax()], color='red', linestyle='--')
plt.tight_layout()
plt.show()
print(f'Best max_depth: {param_range[val_scores.mean(axis=1).argmax()]}')

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(2, 30),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist, n_iter=50, cv=5, scoring='accuracy',
    random_state=42, n_jobs=-1
)
search.fit(X_scaled, y)

print(f'Best CV accuracy: {search.best_score_:.4f}')
print(f'Best params: {search.best_params_}')

## 6. Feature Importance Across Methods

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif

# Train a model
rf = RandomForestClassifier(n_estimators=200, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
rf.fit(X_tr, y_tr)

# Method 1: Built-in feature importance (Gini)
gini_imp = rf.feature_importances_

# Method 2: Permutation importance
perm_imp = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=42)

# Method 3: Mutual information
mi_scores = mutual_info_classif(X_scaled, y, random_state=42)

# Compare top features
top_n = 10
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, imp, title in zip(axes, 
    [gini_imp, perm_imp.importances_mean, mi_scores],
    ['Gini Importance', 'Permutation Importance', 'Mutual Information']):
    idx = np.argsort(imp)[-top_n:]
    ax.barh([data.feature_names[i] for i in idx], imp[idx])
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 7. Model Selection Workflow

In [ ]:
from sklearn.model_selection import learning_curve

# Learning curves to diagnose bias/variance
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, [
    ('Logistic (high bias)', LogisticRegression(C=0.01, max_iter=1000)),
    ('RF (balanced)', RandomForestClassifier(max_depth=10, random_state=42)),
    ('DT (high variance)', DecisionTreeClassifier(random_state=42)),
]):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_scaled, y, train_sizes=np.linspace(0.1, 1.0, 8),
        cv=5, scoring='accuracy', n_jobs=-1
    )
    ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train')
    ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Val')
    ax.fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1),
                    val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.2)
    ax.set_title(name)
    ax.set_xlabel('Training Size')
    ax.set_ylabel('Accuracy')
    ax.legend(loc='lower right')
    ax.set_ylim(0.85, 1.02)

plt.tight_layout()
plt.show()

## 8. Cross-Validation Variants

In [ ]:
from sklearn.model_selection import (
    KFold, StratifiedKFold, RepeatedStratifiedKFold,
    LeaveOneOut, ShuffleSplit
)

model = RandomForestClassifier(n_estimators=100, random_state=42)

cv_strategies = {
    'KFold(5)': KFold(5, shuffle=True, random_state=42),
    'StratifiedKFold(5)': StratifiedKFold(5, shuffle=True, random_state=42),
    'Repeated(5x3)': RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42),
    'ShuffleSplit(10)': ShuffleSplit(n_splits=10, test_size=0.2, random_state=42),
}

for name, cv in cv_strategies.items():
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
    print(f'{name:<25} Accuracy: {scores.mean():.4f} +/- {scores.std():.4f} (n_splits={len(scores)})')

## 9. Calibration and Threshold Tuning

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import precision_recall_curve, f1_score

# Train model
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)

# Calibration
calibrated = CalibratedClassifierCV(rf, method='isotonic', cv=5)
calibrated.fit(X_tr, y_tr)

# Plot calibration
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, model_to_plot in [('Uncalibrated', rf), ('Calibrated', calibrated)]:
    probs = model_to_plot.predict_proba(X_te)[:, 1]
    fraction_pos, mean_pred = calibration_curve(y_te, probs, n_bins=10)
    axes[0].plot(mean_pred, fraction_pos, 's-', label=name)

axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Fraction Positive')
axes[0].set_title('Calibration Curve')
axes[0].legend()

# Threshold tuning
probs = rf.predict_proba(X_te)[:, 1]
thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores = [f1_score(y_te, (probs >= t).astype(int)) for t in thresholds]

axes[1].plot(thresholds, f1_scores, 'o-')
best_thresh = thresholds[np.argmax(f1_scores)]
axes[1].axvline(best_thresh, color='red', linestyle='--', label=f'Best: {best_thresh:.2f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Threshold vs F1')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Optimal threshold: {best_thresh:.2f}, F1: {max(f1_scores):.4f}')

## 10. Deployment-Ready Pipeline

In [ ]:
import joblib
from sklearn.pipeline import Pipeline

# Build final production pipeline
production_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('log_transform', LogTransformer(threshold=1.5)),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
    ))
])

# Final training on all data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
production_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = production_pipeline.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Test ROC-AUC: {roc_auc_score(y_test, production_pipeline.predict_proba(X_test)[:,1]):.4f}')

# Save
joblib.dump(production_pipeline, 'production_model.joblib', compress=3)
print(f'\nModel saved. Usage:')
print('  model = joblib.load("production_model.joblib")')
print('  prediction = model.predict(new_data)')

In [ ]:
# Inference function
def predict_with_validation(model_path, input_array):
    """Production inference with validation."""
    model = joblib.load(model_path)
    X = np.array(input_array)
    if X.ndim == 1:
        X = X.reshape(1, -1)
    if X.shape[1] != 30:
        raise ValueError(f'Expected 30 features, got {X.shape[1]}')
    pred = model.predict(X)
    prob = model.predict_proba(X)
    return {'prediction': pred.tolist(), 'probability': prob.tolist()}

# Test
sample = X_test[0]
result = predict_with_validation('production_model.joblib', sample)
print(f'Prediction: {"Benign" if result["prediction"][0] == 1 else "Malignant"}')
print(f'Confidence: {max(result["probability"][0]):.2%}')